# CSIT 595 Assignment 1: modular Fashion-MNIST comparison

This notebook imports the model implementations from the Python modules in this folder.
Set `CONFIG["full_run"] = False` for a smoke test or `True` for the final GPU experiment.


In [ ]:
import pandas as pd
import torch

from assignment1_common import (
    CONFIG, DEVICE, OUT, EvaluatorCNN, configure_run, evaluator_accuracy,
    predicted_histogram, show_grid, test_loader, train_evaluator, evaluate_model,
)
from assignment1_vae import VAE, train_vae, sample_vae
from assignment1_gan import Generator, Discriminator, train_gan, sample_gan
from assignment1_diffusion import TinyUNet, train_diffusion, sample_diffusion

# Change this one flag before running. The model modules contain the implementation.
CONFIG["full_run"] = True
configure_run(CONFIG["full_run"])


def run_experiment():
    print("Training evaluator...")
    evaluator = EvaluatorCNN().to(DEVICE)
    evaluator, evaluator_history = train_evaluator(evaluator)
    print("Evaluator test accuracy:", evaluator_accuracy(evaluator))

    print("Training VAE...")
    vae = VAE(latent_dim=64).to(DEVICE)
    vae, vae_history, vae_seconds = train_vae(vae)

    print("Training GAN...")
    generator = Generator(latent_dim=128).to(DEVICE)
    discriminator = Discriminator().to(DEVICE)
    generator, discriminator, gan_history, gan_seconds = train_gan(generator, discriminator)

    print("Training diffusion model...")
    denoiser = TinyUNet(base_channels=32, time_dim=64).to(DEVICE)
    denoiser, diffusion, diffusion_history, diffusion_seconds = train_diffusion(denoiser)

    vae_metrics, _ = evaluate_model(
        "VAE", lambda count: sample_vae(vae, count), evaluator,
        training_seconds=vae_seconds, parameters=sum(p.numel() for p in vae.parameters()),
    )
    gan_metrics, _ = evaluate_model(
        "GAN", lambda count: sample_gan(generator, count), evaluator,
        training_seconds=gan_seconds,
        parameters=sum(p.numel() for p in generator.parameters()) + sum(p.numel() for p in discriminator.parameters()),
    )
    ddpm_metrics, _ = evaluate_model(
        "DDPM", lambda count: sample_diffusion(diffusion, denoiser, count, steps=CONFIG["diffusion_steps"]), evaluator,
        training_seconds=diffusion_seconds, parameters=sum(p.numel() for p in denoiser.parameters()),
    )
    results = pd.DataFrame([vae_metrics, gan_metrics, ddpm_metrics])
    results.to_csv(OUT / "results.csv", index=False)
    pd.DataFrame(evaluator_history).to_csv(OUT / "evaluator_history.csv", index=False)
    pd.DataFrame(vae_history).to_csv(OUT / "vae_history.csv", index=False)
    pd.DataFrame(gan_history).to_csv(OUT / "gan_history.csv", index=False)
    pd.DataFrame(diffusion_history).to_csv(OUT / "diffusion_history.csv", index=False)
    print(results)

    preview, _ = next(iter(test_loader))
    preview = preview[:10].to(DEVICE)
    with torch.no_grad():
        reconstruction, _, _ = vae(preview)
        z0, z1 = torch.randn(1, vae.latent_dim, device=DEVICE), torch.randn(1, vae.latent_dim, device=DEVICE)
        vae_interp = torch.cat([vae.decode(torch.lerp(z0, z1, f.item())) for f in torch.linspace(0, 1, 8)])
        z0, z1 = torch.randn(1, generator.latent_dim, device=DEVICE), torch.randn(1, generator.latent_dim, device=DEVICE)
        gan_interp = torch.cat([generator(torch.lerp(z0, z1, f.item())) for f in torch.linspace(0, 1, 8)]).cpu()
    show_grid(torch.cat([preview.cpu(), reconstruction.cpu()]), "VAE originals and reconstructions", "vae_reconstruction.png")
    show_grid(vae_interp.cpu(), "VAE latent interpolation", "vae_interpolation.png", columns=8)
    show_grid(gan_interp, "GAN latent interpolation", "gan_interpolation.png", columns=8)
    show_grid(sample_vae(vae, CONFIG["sample_count"]), "VAE prior samples", "vae_samples.png")
    show_grid(sample_gan(generator, CONFIG["sample_count"]), "GAN samples", "gan_samples.png")
    show_grid(sample_diffusion(diffusion, denoiser, CONFIG["sample_count"], steps=min(CONFIG["diffusion_steps"], 100)), "DDPM samples", "diffusion_samples.png")
    print("Saved artifacts to:", OUT)


if __name__ == "__main__":
    run_experiment()
